#GenAi Lab 8

**Name : Atharv Bhosale**


**PRN : 2324000461**

**Roll No : DS20**

### Step 1: Package Installation
In this step, we install the necessary Python packages, including `openai` (for the API client) and `ddgs` (for DuckDuckGo web searches).

In [ ]:
# Install the small set of packages used by this notebook.
%pip install -q -U openai ddgs


### Step 2: Import Packages and Load API Key
Here we import all the required libraries and securely load the Groq API key from Colab's Secrets manager to configure our client.

In [ ]:
# Import packages and read the API key securely from Colab Secrets.
from google.colab import userdata
from openai import OpenAI
from ddgs import DDGS
import ast
import operator
import json
import math
import os

api_key = userdata.get("GROQ_API_KEY")
if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Open Colab Secrets, add GROQ_API_KEY, "
        "and enable notebook access."
    )

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
print("Imports complete and API key loaded securely.")


Imports complete and API key loaded securely.


### Step 3: Model Selection
We inspect the available models offered by Groq and choose a powerful, tool-capable model (like `openai/gpt-oss-120b`) for our agent.

In [ ]:
# Find available Groq models and select a model that supports tool use.
model_ids = sorted(model.id for model in client.models.list().data)
print("Models available to this API key:")
for model_id in model_ids:
    print(" -", model_id)

preferred_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]
MODEL = next((m for m in preferred_models if m in model_ids), None)

# If the preferred list does not match, try an available GPT-OSS model.
if MODEL is None:
    MODEL = next((m for m in model_ids if "gpt-oss" in m.lower()), None)

if MODEL is None:
    raise RuntimeError(
        "No model from the notebook's preferred tool-capable list was found. "
        "Review the printed model IDs and set MODEL to a chat model that supports "
        "tool calling in the next cell."
    )

print("\nSelected model:", MODEL)


Models available to this API key:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo

Selected model: openai/gpt-oss-120b


### Step 4: Define the Calculator Tool
This cell defines a secure math calculator using Python's Abstract Syntax Trees (`ast`) to safely evaluate arithmetic expressions without using `eval()`.

In [ ]:
# Safely evaluate basic arithmetic without Python eval().
_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

def _safe_math(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_math(node.left)
        right = _safe_math(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("Exponent is too large.")
        result = _ALLOWED_OPERATORS[type(node.op)](left, right)
        if isinstance(result, (int, float)) and not math.isfinite(result):
            raise ValueError("Result is not finite.")
        return result
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_math(node.operand))
    raise ValueError("Only basic arithmetic with numbers and parentheses is allowed.")

def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_math(tree.body)
        return str(result)
    except Exception as exc:
        return f"Calculator error: {exc}"

print("Calculator test:", calculator("342 * 19"))


Calculator test: 6498


### Step 5: Define the Web Search Tool
We implement a web search helper using DuckDuckGo Search (`DDGS`) to fetch real-time facts and snippets from the internet.

In [ ]:
# Search the live web using DuckDuckGo.
def web_search(query: str) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No search results were returned."
        compact_results = []
        for item in results:
            compact_results.append({
                "title": item.get("title", ""),
                "url": item.get("href", ""),
                "snippet": item.get("body", "")
            })
        return json.dumps(compact_results, ensure_ascii=False)
    except Exception as exc:
        return f"Web search error: {type(exc).__name__}: {exc}"

print("Web search tool is ready.")


Web search tool is ready.


### Step 6: Tool Binding and Registry
Here we register the schemas for our two tools (`Calculator` and `WebSearch`) in the format expected by the model's function calling API.

In [ ]:
# Describe the two tools the model may choose to call.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "Calculator",
            "description": "Evaluate a basic arithmetic expression, for example 342 * 19. Use this for calculations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Arithmetic expression containing numbers and basic operators."
                    }
                },
                "required": ["expression"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "WebSearch",
            "description": "Search the live web for current facts, news, and information that may have changed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A concise search query."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

def run_tool(name, arguments):
    if name == "Calculator":
        return calculator(arguments["expression"])
    if name == "WebSearch":
        return web_search(arguments["query"])
    return f"Unknown tool: {name}"

print("Registered tools:", ", ".join(t["function"]["name"] for t in TOOLS))


Registered tools: Calculator, WebSearch


### Step 7: Define the Agent ReAct Loop
This function coordinates the execution loop where the LLM can request tool executions, parse their outcomes, update its memory, and reason iteratively until it produces a final answer.

In [ ]:
# Run an agent loop: the model may request tools, inspect results, and answer.
def run_agent(question: str, max_tool_rounds: int = 5, verbose: bool = True) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant completing an Agentic AI practical. "
                "Use Calculator for arithmetic and WebSearch for current or changing facts. "
                "When a tool returns an observation, use it to answer accurately. "
                "Do not claim to have searched if WebSearch was not called. "
                "Give a concise final answer and distinguish sourced facts from uncertainty."
            )
        },
        {"role": "user", "content": question}
    ]

    for round_number in range(max_tool_rounds + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0
        )
        message = response.choices[0].message
        messages.append(message)

        tool_calls = message.tool_calls or []
        if not tool_calls:
            return message.content or "(The model returned no text answer.)"

        if verbose:
            print(f"\nAgent requested {len(tool_calls)} tool call(s):")

        for tool_call in tool_calls:
            name = tool_call.function.name
            try:
                arguments = json.loads(tool_call.function.arguments or "{}")
            except json.JSONDecodeError:
                arguments = {}

            if verbose:
                print(f"  Action: {name}")
                print(f"  Action Input: {arguments}")

            observation = run_tool(name, arguments)

            if verbose:
                print(f"  Observation: {observation[:1200]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(observation)
            })

    # One final request without tools avoids an endless tool loop.
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0
    )
    return final_response.choices[0].message.content or "No final answer returned."

print("Agent function is ready.")


Agent function is ready.


### Step 8: Main Experiment (Multi-Tool Query)
We execute a hybrid prompt requiring both math calculations and live web search to test if our agent orchestrates multiple tools correctly.

In [ ]:
# Main experiment: calculate 342 * 19 and search for India's current Prime Minister.
question = "What is 342 * 19, and who is the current Prime Minister of India?"
answer = run_agent(question, max_tool_rounds=5, verbose=True)

print("\nFinal Answer:")
print(answer)



Agent requested 1 tool call(s):
  Action: Calculator
  Action Input: {'expression': '342 * 19'}
  Observation: 6498

Agent requested 1 tool call(s):
  Action: WebSearch
  Action Input: {'query': 'current Prime Minister of India'}
  Observation: [{"title": "Narendra Modi - Wikipedia", "url": "https://en.wikipedia.org/wiki/Narendra_Modi", "snippet": "India's president appointed Modi, who was unanimously elected leader of the BJP, Prime Minister of India. [206][207] To comply with the law prohibiting MPs from representing more than one constituency, he vacated the Vadodara seat. [208]"}, {"title": "Prime Minister of India - Wikipedia", "url": "https://en.wikipedia.org/wiki/Prime_minister_of_india", "snippet": "India follows a parliamentary system in which the prime minister is the presiding head of the government and chief of the executive of the government. In such systems, the head of state, or, the head of state's official representative (i.e., the monarch, president, or governor-gene

### Step 9: Optional Test 1 (Arithmetic Only)
We run a test query that requires only mathematical calculations to verify if the agent correctly ignores the search tool when it is not needed.

In [ ]:
# Optional test 1: arithmetic-only question should use Calculator.
answer_math = run_agent("Calculate 125 * 48.", verbose=True)
print("\nFinal Answer:", answer_math)



Agent requested 1 tool call(s):
  Action: Calculator
  Action Input: {'expression': '125 * 48'}
  Observation: 6000

Final Answer: **Result:** 125 × 48 = **6000**.


### Step 10: Optional Test 2 (Current Information Only)
We request current factual news from 2024 to verify if the agent successfully prioritizes the Web Search tool for dynamic updates.

In [ ]:
# Optional test 2: current information should use WebSearch.
answer_current = run_agent(
    "Find a recent reliable source confirming who is currently the Prime Minister of India.",
    verbose=True
)
print("\nFinal Answer:", answer_current)



Agent requested 1 tool call(s):
  Action: WebSearch
  Action Input: {'query': 'current Prime Minister of India news 2024'}
  Observation: Web search error: DDGSException: No results found.

Agent requested 1 tool call(s):
  Action: WebSearch
  Action Input: {'query': 'Prime Minister of India 2024'}
  Observation: [{"title": "2024 Indian general election - WikipediaList of prime ministers of India - WikipediaPrime Minister of IndiaKnow the PM | Prime Minister of IndiaList of All Prime Ministers of India (1947-2026) - Jagran JoshList of all Prime Ministers of India: 1947 to 2024List of Prime Ministers of India, 1947-2026, Tenure, Party", "url": "https://en.wikipedia.org/wiki/2024_Indian_general_election", "snippet": "The politics of India became increasingly bipolar in the run-up to the 2024 Indian general elections with two major alliances emerging; the incumbent National Democratic Alliance (NDA) and the opposition Indian National Developmental Inclusive Alliance (INDIA). Following th

### Step 11: Optional Test 3 (General Knowledge)
We ask a static general knowledge question to test if the agent can answer directly from its pre-trained weights without calling any tools.

In [ ]:
# Optional test 3: general knowledge question.
answer_general = run_agent("What is the capital of France?", verbose=True)
print("\nFinal Answer:", answer_general)



Final Answer: The capital of France is **Paris**.


## Observations and Summary
1. **Tool-Use Accuracy:** The model (`openai/gpt-oss-120b`) successfully reasoned about when to use the `Calculator` (for math) and when to use `WebSearch` (for dynamic facts like the Prime Minister).
2. **Zero-Shot General Knowledge:** For basic facts (e.g., "What is the capital of France?"), the agent correctly avoided unnecessary tool invocations.
3. **Robustness:** Using `ast` parsing prevents arbitrary code execution, keeping our calculator tool secure and isolated.